# Exploring RMD API Endpoints
Let's look at what data is available from the PSU Researcher Metadata Database.

In [1]:
import requests
import json

RMD_BASE = "https://metadata.libraries.psu.edu/v1"
API_KEY = "264d2e02f4c8089b594de2ad8db5f80c0ada5828deca6f06a4d5690d4cd80f869e44aa32b7e4dead64a56f963b368be7"
AUTH_HEADERS = {"X-API-Key": API_KEY, "Accept": "application/json"}
PUBLIC_HEADERS = {"Accept": "application/json"}

def rmd_get(path, params=None, auth=True):
    headers = AUTH_HEADERS if auth else PUBLIC_HEADERS
    resp = requests.get(f"{RMD_BASE}{path}", headers=headers, params=params or {})
    print(f"{resp.status_code} {resp.url}")
    if resp.ok:
        return resp.json()
    print(f"  Error: {resp.text[:200]}")
    return None

## 1. Organizations (HHD)

In [2]:
orgs = rmd_get("/organizations")
for org in orgs.get("data", []):
    print(f"  ID {org['id']}: {org.get('attributes', {}).get('name', '?')}")

200 https://metadata.libraries.psu.edu/v1/organizations
  ID 10: College of Health and Human Development
  ID 182: Center For Healthy Aging
  ID 183: Methodology Center
  ID 184: Edna Bennett Pierce Prevention Research Center (PRC)
  ID 185: Biobehavioral Health
  ID 186: Communication Sciences and Disorders
  ID 187: Human Development and Family Studies
  ID 188: Kinesiology
  ID 189: Nutritional Sciences
  ID 190: Recreation, Park and Tourism Management
  ID 191: Health Policy and Administration
  ID 192: School of Hospitality Management
  ID 511: College of Health and Human Development - Administration
  ID 565: College of Health and Human Development Research Centers
  ID 587: Biomarker Core Lab
  ID 809: Child Health Research Center
  ID 810: Center for Health Care and Policy Research


## 2. Single Organization — what fields are available?

In [3]:
# College of HHD = org ID 10
org_detail = rmd_get("/organizations/10")
print(json.dumps(org_detail, indent=2)[:2000])

404 https://metadata.libraries.psu.edu/v1/organizations/10
  Error: {"status":404,"error":"Not Found"}
null


## 3. Org Publications — what does a single publication look like?

In [4]:
pubs = rmd_get("/organizations/10/publications", params={"limit": 3})
if pubs:
    for pub in pubs.get("data", []):
        attrs = pub.get("attributes", {})
        print(f"\nPub ID: {pub.get('id')}")
        print(f"  Title: {attrs.get('title', '?')[:80]}")
        print(f"  DOI: {attrs.get('doi')}")
        print(f"  Contributors:")
        for c in attrs.get("contributors", []):
            print(f"    {c.get('first_name')} {c.get('last_name')} (user_id: {c.get('psu_user_id')}, orcid: {c.get('orcid')})")
        print(f"  All keys: {list(attrs.keys())}")

200 https://metadata.libraries.psu.edu/v1/organizations/10/publications?limit=3

Pub ID: 4
  Title: Rapid estrogen receptor-α activation improves ischemic tolerance in aged female 
  DOI: https://doi.org/10.1210/en.2008-0708
  Contributors:
    Jennifer L. Novotny (user_id: None, orcid: None)
    Amy M. Simpson (user_id: None, orcid: None)
    Nanette J. Tomicek (user_id: None, orcid: None)
    Timothy S. Lancaster (user_id: None, orcid: None)
    Donna H. Korzick (user_id: dhk102, orcid: None)
  All keys: ['title', 'secondary_title', 'publication_type', 'status', 'volume', 'issue', 'edition', 'page_range', 'authors_et_al', 'abstract', 'doi', 'preferred_open_access_url', 'publisher', 'journal_title', 'published_on', 'citation_count', 'supplementary_url', 'contributors', 'tags', 'pure_ids', 'activity_insight_ids', 'profile_preferences']

Pub ID: 13
  Title: Determinants of water and sodium intake and output
  DOI: https://doi.org/10.1093/nutrit/nuv033
  Contributors:
    Anna E. Stanhew

## 4. User Profile — what fields does a profile have?

In [5]:
# Pick a known user from our map
profile = rmd_get("/users/dej10/profile", auth=False)
if profile:
    attrs = profile.get("data", {}).get("attributes", {})
    print("Profile fields:")
    for k, v in attrs.items():
        val_str = str(v)[:100] if v else "None"
        print(f"  {k}: {val_str}")

200 https://metadata.libraries.psu.edu/v1/users/dej10/profile
Profile fields:
  name: Damon Evan Jones
  organization_name: Edna Bennett Pierce Prevention Research Center (PRC)
  title: Research Professor
  office_location: 316B 316 B Biobehavioral Health Building
  office_phone_number: (814) 865-6020
  personal_website: None
  total_scopus_citations: 3809
  scopus_h_index: 28
  pure_profile_url: https://pure.psu.edu/en/persons/8b33cf93-dba9-40ae-980b-ad43c63c6600
  orcid_identifier: None
  bio: Damon Jones is a researcher and instructor in the College of Health and Human Development.  His prim
  teaching_interests: Statistics in the behavioral sciences
Economic evaluation of effective social policy programs
  research_interests: Methodology to improve analysis in the behavioral sciences, Economic evaluation of social policy int
  publications: ['<span class="publication-title"><a href="https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/cde
  other_publications: {'Blogs': ['<span cl

## 5. Check if publications have ORCIDs on contributors

In [6]:
# Fetch a larger batch and check how many contributors have orcid fields
pubs = rmd_get("/organizations/10/publications", params={"limit": 100})
total_contribs = 0
with_orcid = 0
with_user_id = 0
sample_orcids = []

if pubs:
    for pub in pubs.get("data", []):
        for c in pub.get("attributes", {}).get("contributors", []):
            total_contribs += 1
            if c.get("psu_user_id"):
                with_user_id += 1
            if c.get("orcid"):
                with_orcid += 1
                if len(sample_orcids) < 5:
                    sample_orcids.append((c.get('first_name'), c.get('last_name'), c.get('orcid'), c.get('psu_user_id')))

print(f"Total contributors: {total_contribs}")
print(f"With psu_user_id: {with_user_id}")
print(f"With orcid: {with_orcid}")
print(f"\nSample contributors with ORCIDs:")
for name_f, name_l, orcid, uid in sample_orcids:
    print(f"  {name_f} {name_l} — ORCID: {orcid}, user_id: {uid}")

200 https://metadata.libraries.psu.edu/v1/organizations/10/publications?limit=100
Total contributors: 393
With psu_user_id: 98
With orcid: 0

Sample contributors with ORCIDs:


## 6. Check all contributor fields on a single publication

In [ ]:
# Show raw contributor JSON to see ALL available fields
if pubs:
    for pub in pubs.get("data", []):
        contribs = pub.get("attributes", {}).get("contributors", [])
        if contribs:
            print("Contributor keys:", list(contribs[0].keys()))
            print("\nFirst 3 contributors (raw):")
            for c in contribs[:3]:
                print(json.dumps(c, indent=2))
            break

## 7. Check a department (e.g., Kinesiology) for users

In [ ]:
# Kinesiology = org 188
kin_pubs = rmd_get("/organizations/188/publications", params={"limit": 50})
kin_users = set()
kin_orcids = {}
if kin_pubs:
    for pub in kin_pubs.get("data", []):
        for c in pub.get("attributes", {}).get("contributors", []):
            uid = c.get("psu_user_id")
            orcid = c.get("orcid")
            if uid:
                kin_users.add(uid)
                if orcid:
                    kin_orcids[uid] = orcid

print(f"Kinesiology unique users: {len(kin_users)}")
print(f"With ORCID on publication: {len(kin_orcids)}")
print(f"\nUsers with ORCIDs:")
for uid, orcid in list(kin_orcids.items())[:10]:
    print(f"  {uid}: {orcid}")

## 8. Try other endpoints — users list? search?

In [ ]:
# Try some endpoints that might list users directly
for path in ["/users", "/organizations/10/users", "/organizations/10/members"]:
    print(f"\nTrying {path}...")
    result = rmd_get(path)
    if result:
        data = result.get("data", [])
        print(f"  Got {len(data)} results")
        if data:
            print(f"  First item keys: {list(data[0].keys()) if isinstance(data[0], dict) else type(data[0])}")
            print(f"  Sample: {json.dumps(data[0], indent=2)[:300]}")

## 9. Summary: Where can we get ORCIDs?

In [ ]:
print("Sources of ORCIDs in RMD:")
print("1. Publication contributors — 'orcid' field (check cell 5 results)")
print("2. User profiles — 'orcid_identifier' field (what we currently use)")
print("")
print("If publications have ORCIDs directly on contributors,")
print("we can skip the expensive per-user profile lookup entirely.")